# Assignement 1 - SD - 09 05 2025

### 1. Installation des outils

In [1]:
import os
from datetime import datetime

from openhexa.sdk import workspace
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import (
get_organisation_units,          
get_organisation_unit_groups,    
get_data_element_groups,
get_data_elements,
extract_dataset,
extract_data_element_group,
extract_events)

from openhexa.toolbox.dhis2.dataframe import get_datasets

### 2. Connection to dhis2-demo

In [2]:
from dotenv import load_dotenv
import os
# Charger ton fichier
load_dotenv("variable.env")
print("DHIS2_DEMO =", os.environ.get("DHIS2_DEMO"))
print("DHIS2_DEMO_TYPE =", os.environ.get("DHIS2_DEMO_TYPE"))

DHIS2_DEMO = dhis2
DHIS2_DEMO_TYPE = dhis2


In [3]:
con = workspace.dhis2_connection("dhis2-demo")
sle = DHIS2(con, cache_dir="/tmp/.cache")
sle.api.session.auth = (con.username, con.password)
#sle = DHIS2(con, cache_dir="/home/hexa/workspace/.cache")

### 3. Définition des paramètres

In [4]:
niveau_analyse = 4                                                 #Help message : Varie entre 1 et 4 pour DHIS2 demo
groupe_analyse = ["Public facilities","Private Clinic"]            #Help message : Voir liste avec le nom des groupes plus bas           
niveau_agregation = 3                                              #Help message : Varie entre 1 et 4 pour DHIS2 demo

if niveau_agregation > niveau_analyse:
    
    print("Attention, parramètres incorrects!")
else:
    print("Parramètres saisis semblent ok!")                       # Vérification supplémentaires possibles

Parramètres saisis semblent ok!


### 4. Obtention du fichier orgunits complet

In [5]:
df = get_organisation_units(sle)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
print(type(df))
print(len(df))
ou = df.to_pandas()
ou["level"].value_counts().sort_index()

<class 'polars.dataframe.frame.DataFrame'>
1332


level
1       1
2      13
3     152
4    1166
Name: count, dtype: int64

In [ ]:
oug = get_organisation_unit_groups(sle)
oug_pd = oug.to_pandas()
oug_pd['ou_count']=oug_pd['organisation_units'].apply(len)
oug_pd.head(100)

,id,name,organisation_units,ou_count
0,CXw2yu5fodb,CHC,"[Mi4dWRtfIOC, EUUkKEDoNsf, qVvitxEF2ck, HWXk4E...",194
1,uYxK4wmcPqA,CHP,"[EQnfnY03sRp, ZpE2POxvl9P, n2qFnUIhbq3, f90eIS...",223
2,gzcv65VyaGq,Chiefdom,"[nV3OkyzF4US, r06ohri9wA9, Z9QaI6sxTwW, A3Fh37...",152
3,RXL3lPSK8oG,Clinic,"[LaxJ6CD2DHq, ctfiYW0ePJ8, M721NHGtdZV, yP2nhl...",51
4,RpbiCJpIYEj,Country,[ImspTQPwCqd],1
5,w1Atoz18PCL,District,"[at6UHUQatSo, TEQlaapDQoK, PMa2VCrupOd, kJq2mP...",13
6,nlX2VoouN63,Eastern Area,"[PMa2VCrupOd, bL4ooGhyHRQ, jUb8gELQApl]",3
7,tDZVQ1WtwpA,Hospital,"[U8uqyDAu5bH, BV4IomHvri4, rIgJX4N0DGZ, OzjRQL...",34
8,EYbopBOJWsW,MCHP,"[AFi1GjbeejL, y77LiPqLMoq, ALZ2qr5u0X0, AiGBOD...",594
9,w0gFTTmsUcF,Mission,"[Tht0fnjagHi, Z9ny6QeqsgX, VrDA0Hn4Xc6, bqtZrX...",6


In [ ]:
# Étape 1 : Transformer organisation_units (listes) en lignes individuelles
df_exploded = oug_pd.explode("organisation_units")
# Étape 2 : Créer un dictionnaire {id_org: [groupes]} pour savoir à quels groupes chaque ID appartient
group_mapping = df_exploded.groupby("organisation_units")["name"].apply(list).to_dict()
# Étape 3 : Ajouter une colonne par groupe dans org_units_pd
for group in oug_pd["name"]:
    # Vérifier si chaque ID est dans le groupe et remplir 1 ou 0
    ou[group] = ou["id"].apply(lambda x: 1 if group in group_mapping.get(x, []) else 0)

# Affichage du DataFrame mis à jour
ou.head(10)

,id,name,level,opening_date,closed_date,level_1_id,level_1_name,level_2_id,level_2_name,level_3_id,...,MCHP,Mission,NGO,Northern Area,Private Clinic,Public facilities,Rural,Southern Area,Urban,Western Area
0,ImspTQPwCqd,Sierra Leone,1,1994-01-01,NaT,ImspTQPwCqd,Sierra Leone,None,None,None,...,0,0,0,0,0,0,0,0,0,0
1,O6uvpzGd5pu,Bo,2,1990-02-01,NaT,ImspTQPwCqd,Sierra Leone,O6uvpzGd5pu,Bo,None,...,0,0,0,0,0,0,0,1,0,0
2,fdc6uOvgoji,Bombali,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,fdc6uOvgoji,Bombali,None,...,0,0,0,1,0,0,1,0,0,0
3,lc3eMKXaEfw,Bonthe,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,lc3eMKXaEfw,Bonthe,None,...,0,0,0,0,0,0,1,1,0,0
4,jUb8gELQApl,Kailahun,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,jUb8gELQApl,Kailahun,None,...,0,0,0,0,0,0,1,0,0,0
5,PMa2VCrupOd,Kambia,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,PMa2VCrupOd,Kambia,None,...,0,0,0,0,0,0,1,0,0,1
6,kJq2mPyFEHo,Kenema,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,kJq2mPyFEHo,Kenema,None,...,0,0,0,0,0,0,0,0,1,0
7,qhqAxPSTUXp,Koinadugu,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,qhqAxPSTUXp,Koinadugu,None,...,0,0,0,1,0,0,0,0,1,0
8,Vth0fbpFcsO,Kono,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,Vth0fbpFcsO,Kono,None,...,0,0,0,1,0,0,0,0,1,0
9,jmIPBj66vD6,Moyamba,2,1970-01-01,NaT,ImspTQPwCqd,Sierra Leone,jmIPBj66vD6,Moyamba,None,...,0,0,0,0,0,0,0,1,1,0


In [ ]:
ou["has_geometry"] = ou["geometry"].notna()
ou["missing_geometry"] = ou["geometry"].isna()

In [ ]:
# Filtrer le DataFrame en fonction du niveau 
ou_selected = ou[(ou["level"] == niveau_analyse)]

# Filtrer le DataFrame en fonction des groupes (colonnes booléennes)
ou_selected = ou_selected[ou_selected[groupe_analyse].any(axis=1)]


In [ ]:
lvl = f"level_{niveau_agregation}_id"
result = ou_selected.groupby(lvl).agg({
    "has_geometry": "sum",  # Somme des 1 pour le nombre de lignes avec géométrie
    "missing_geometry": "sum", # Nombre de lignes sans géométrie
}).reset_index()

# Calculer le pourcentage correctement (en utilisant les deux colonnes)
result["percent_geometry"] = (result["has_geometry"] / (result["has_geometry"] + result["missing_geometry"])) * 100
result["percent_geometry"] = result["percent_geometry"].round(1)

In [ ]:
ou_reduced = ou.drop_duplicates(subset=[lvl], keep='first')[['level_1_name', 'level_2_name', 'level_3_name', lvl]]

result_with_names = result.merge(
    ou_reduced,
    on=lvl,
    how='left'
)

In [ ]:
# Obtenir la date du jour au format YYYY-MM-DD
date_today = datetime.now().strftime("%Y-%m-%d")

# Vérifier s'il existe déjà un fichier avec ce nom et ajouter un numéro séquentiel
counter = 1
file_name = f"output_{date_today}_{counter}.csv"

# Si le fichier existe déjà, incrémenter le numéro
while os.path.exists(file_name):
    counter += 1
    file_name = f"output_{date_today}_{counter}.csv"

# Enregistrer le DataFrame résultant dans ce fichier CSV
result_with_names.to_csv(file_name, index=False)

# Afficher le nom du fichier créé
print(f"Fichier sauvegardé sous : {file_name} in {os.getcwd()}")

Fichier sauvegardé sous : output_2025-05-12_2.csv in /home/jovyan/workspace/sd/Assignement_1


### Liste de questions et commentaire